In [3]:

import os 
import pandas as pd
import json
from sqlalchemy import create_engine
import sqlite3

In [4]:
engine = create_engine("sqlite:///Restautrant.db")

In [3]:
folder_path = r"C:\Users\satvi\Downloads\Yelp-JSON\Yelp JSON\yelp_dataset"

for file in os.listdir(folder_path):
    if file.endswith('.json'):
        table_name = file.replace('.json','')
        print(f"Loading {file}.....")
        
        first_chunk = True
        
        for chunk in pd.read_json(
            os.path.join(folder_path,file),
            lines=True,
            chunksize = 100000
        ):
            for col in chunk.columns:
                if chunk[col].apply(lambda x: isinstance(x,dict)).any():
                    chunk[col] = chunk[col].apply(
                        lambda x: json.dumps(x) if isinstance(x,dict) else x
                    )
                
            chunk.to_sql(
                table_name,
                engine,
                if_exists = 'replace' if first_chunk else 'append',
                index = False
            )
            
            first_chunk = False
        
        print(f"loaded {table_name}")

Loading yelp_academic_dataset_business.json.....
loaded yelp_academic_dataset_business
Loading yelp_academic_dataset_checkin.json.....
loaded yelp_academic_dataset_checkin
Loading yelp_academic_dataset_review.json.....
loaded yelp_academic_dataset_review
Loading yelp_academic_dataset_tip.json.....
loaded yelp_academic_dataset_tip
Loading yelp_academic_dataset_user.json.....
loaded yelp_academic_dataset_user


In [5]:
conn = sqlite3.connect('Restautrant.db')

In [6]:
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type = 'table'",conn)

In [7]:
for table in tables['name']:
    display(pd.read_sql(
        f"SELECT * FROM {table} LIMIT 5",
        conn
))

,business_id,name,address,city,state,postal_code,latitude,longitude,stars,review_count,is_open,attributes,categories,hours
0,Pns2l4eNsfO8kk83dixA6A,"Abby Rappoport, LAC, CMQ","1616 Chapala St, Ste 2",Santa Barbara,CA,93101,34.426679,-119.711197,5.0,7,0,"{""ByAppointmentOnly"": ""True""}","Doctors, Traditional Chinese Medicine, Naturop...",NaN
1,mpf3x-BjTdTEA3yCZrAYPw,The UPS Store,87 Grasso Plaza Shopping Center,Affton,MO,63123,38.551126,-90.335695,3.0,15,1,"{""BusinessAcceptsCreditCards"": ""True""}","Shipping Centers, Local Services, Notaries, Ma...","{""Monday"": ""0:0-0:0"", ""Tuesday"": ""8:0-18:30"", ..."
2,tUFrWirKiKi_TAnsVWINQQ,Target,5255 E Broadway Blvd,Tucson,AZ,85711,32.223236,-110.880452,3.5,22,0,"{""BikeParking"": ""True"", ""BusinessAcceptsCredit...","Department Stores, Shopping, Fashion, Home & G...","{""Monday"": ""8:0-22:0"", ""Tuesday"": ""8:0-22:0"", ..."
3,MTSW4McQd7CbVtyjqoe9mw,St Honore Pastries,935 Race St,Philadelphia,PA,19107,39.955505,-75.155564,4.0,80,1,"{""RestaurantsDelivery"": ""False"", ""OutdoorSeati...","Restaurants, Food, Bubble Tea, Coffee & Tea, B...","{""Monday"": ""7:0-20:0"", ""Tuesday"": ""7:0-20:0"", ..."
4,mWMc6_wTdE0EUBKIGXDVfA,Perkiomen Valley Brewery,101 Walnut St,Green Lane,PA,18054,40.338183,-75.471659,4.5,13,1,"{""BusinessAcceptsCreditCards"": ""True"", ""Wheelc...","Brewpubs, Breweries, Food","{""Wednesday"": ""14:0-22:0"", ""Thursday"": ""16:0-2..."


,business_id,date
0,---kPU91CF4Lq2-WlRu9Lw,"2020-03-13 21:10:56, 2020-06-02 22:18:06, 2020..."
1,--0iUa4sNDFiZFrAdIWhZQ,"2010-09-13 21:43:09, 2011-05-04 23:08:15, 2011..."
2,--30_8IhuyMHbSOcNWd6DQ,"2013-06-14 23:29:17, 2014-08-13 23:20:22"
3,--7PUidqRWpRSpXebiyxTg,"2011-02-15 17:12:00, 2011-07-28 02:46:10, 2012..."
4,--7jw19RH9JKXgFohspgQw,"2014-04-21 20:42:11, 2014-04-28 21:04:46, 2014..."


,review_id,user_id,business_id,stars,useful,funny,cool,text,date
0,KU_O5udG6zpxOg-VcAEodg,mh_-eMZ6K5RLWhZyISBhwA,XQfwVwDr-v0ZS3_CbbE5Xw,3,0,0,0,"If you decide to eat here, just be aware it is...",2018-07-07 22:09:11.000000
1,BiTunyQ73aT9WBnpR9DZGw,OyoGAe7OKpv6SyGZT5g77Q,7ATYjTIgM3jUlt4UM3IypQ,5,1,0,1,I've taken a lot of spin classes over the year...,2012-01-03 15:28:18.000000
2,saUsX_uimxRlCVr67Z4Jig,8g_iMtfSiwikVnbP2etR0A,YjUWPpI6HXG530lwP-fb2A,3,0,0,0,Family diner. Had the buffet. Eclectic assortm...,2014-02-05 20:30:30.000000
3,AqPFMleE6RsU23_auESxiA,_7bHUi9Uuf5__HHc_Q8guQ,kxX2SOes4o-D3ZQBkiMRfA,5,1,0,1,"Wow! Yummy, different, delicious. Our favo...",2015-01-04 00:01:03.000000
4,Sx8TMOWLNuJBWer-0pcmoA,bcjbaE6dDog4jkNY91ncLQ,e4Vwtrqf-wpJfwesgvdgxQ,4,1,0,1,Cute interior and owner (?) gave us tour of up...,2017-01-14 20:54:15.000000


,user_id,business_id,text,date,compliment_count
0,AGNUgVwnZUey3gcPCJ76iw,3uLgwr0qeCNMjKenHJwPGQ,Avengers time with the ladies.,2012-05-18 02:17:21.000000,0
1,NBN4MgHP9D3cw--SnauTkA,QoezRbYQncpRqyrLH6Iqjg,They have lots of good deserts and tasty cuban...,2013-02-05 18:35:10.000000,0
2,-copOvldyKh1qr-vzkDEvw,MYoRNLb5chwjQe3c_k37Gg,It's open even when you think it isn't,2013-08-18 00:56:08.000000,0
3,FjMQVZjSqY8syIO-53KFKw,hV-bABTK-glh5wj31ps_Jw,Very decent fried chicken,2017-06-27 23:05:38.000000,0
4,ld0AperBXk1h6UbqmM80zw,_uN0OudeJ3Zl_tf6nxg5ww,Appetizers.. platter special for lunch,2012-10-06 19:43:09.000000,0


,user_id,name,review_count,yelping_since,useful,funny,cool,elite,friends,fans,...,compliment_more,compliment_profile,compliment_cute,compliment_list,compliment_note,compliment_plain,compliment_cool,compliment_funny,compliment_writer,compliment_photos
0,qVc8ODYU5SZjKXVBgXdI7w,Walker,585,2007-01-25 16:47:26,7217,1259,5994,2007,"NSCy54eWehBJyZdG2iE84w, pe42u7DcCH2QmI81NX-8qA...",267,...,65,55,56,18,232,844,467,467,239,180
1,j14WgRoU_-2ZE1aw1dXrJg,Daniel,4333,2009-01-25 04:35:42,43091,13066,27281,"2009,2010,2011,2012,2013,2014,2015,2016,2017,2...","ueRPE0CX75ePGMqOFVj6IQ, 52oH4DrRvzzl8wh5UXyU0A...",3138,...,264,184,157,251,1847,7054,3131,3131,1521,1946
2,2WnXYQFK0hXEoTxPtV2zvg,Steph,665,2008-07-25 10:41:00,2086,1010,1003,"2009,2010,2011,2012,2013","LuO3Bn4f3rlhyHIaNfTlnA, j9B4XdHUhDfTKVecyWQgyA...",52,...,13,10,17,3,66,96,119,119,35,18
3,SZDeASXq7o05mMNLshsdIA,Gwen,224,2005-11-29 04:38:33,512,330,299,"2009,2010,2011","enx1vVPnfdNUdPho6PH_wg, 4wOcvMLtU6a9Lslggq74Vg...",28,...,4,1,6,2,12,16,26,26,10,9
4,hA5lMy-EnncsH4JoR-hFGQ,Karen,79,2007-01-05 19:40:59,29,15,7,,"PBK4q9KEEBHhFvSXCUirIw, 3FWPpM7KU1gXeOM_ZbYMbA...",1,...,1,0,0,0,1,1,0,0,0,0


In [7]:
tables 

,name
0,yelp_academic_dataset_business
1,yelp_academic_dataset_checkin
2,yelp_academic_dataset_review
3,yelp_academic_dataset_tip
4,yelp_academic_dataset_user


## Data Anlaysis

In [8]:
pd.read_sql_query("SELECT COUNT(*) FROM yelp_academic_dataset_business",conn)

,COUNT(*)
0,150346


In [9]:
business_id = pd.read_sql_query("""
                  SELECT business_id,
                  review_count
                  FROM yelp_academic_dataset_business 
                  WHERE LOWER(categories) LIKE '%restaurant%' and is_open = 1 
                  """,conn)

In [10]:
# What is the descriptive statistics of the review count and star rating?

pd.read_sql_query(f"""
                  SELECT avg(review_count) as Average,
                  min(review_count) as Minimum,
                  max(review_count) as Maximum,
                  (SELECT review_count FROM yelp_academic_dataset_business ORDER BY review_count LIMIT 1 OFFSET(SELECT COUNT(*)/2 FROM yelp_academic_dataset_business)) AS median,
                  
                  avg(stars) as Rating_Average,
                  min(stars) as Rating_Minimum,
                  max(stars) as Rating_Maximum,
                  (SELECT stars FROM yelp_academic_dataset_business ORDER BY stars LIMIT 1 OFFSET(SELECT COUNT(*)/2 FROM yelp_academic_dataset_business)) AS Rating_median
                    
                  FROM yelp_academic_dataset_business
                  WHERE business_id IN {tuple(business_id['business_id'])}
                  """,conn).transpose()

,0
Average,104.097789
Minimum,5.000000
Maximum,7568.000000
median,15.000000
Rating_Average,3.523969
Rating_Minimum,1.000000
Rating_Maximum,5.000000
Rating_median,3.500000


In [10]:
values = pd.read_sql_query("SELECT review_count FROM yelp_academic_dataset_business",conn)

In [11]:
values['review_count'][values['review_count'] == 15].value_counts()

review_count
15    3258
Name: count, dtype: int64

In [12]:
def remove_outliers(df,col):
    Q1 = df[col].quantile(0.25)
    Q3 = df[col].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    df = df[(df[col] >= lower_bound) & (df[col] <= upper_bound)]
    
    return df

In [13]:
business_id = remove_outliers(business_id, 'review_count')

In [14]:
business_id.shape

(31537, 2)

In [15]:
# which restautrants have the highest review

pd.read_sql_query(f"""SELECT name,
                  SUM(review_count) as total_count,
                  avg(stars) as average_rating
                  FROM yelp_academic_dataset_business
                  WHERE business_id IN {tuple(business_id['business_id'])}
                  GROUP BY name
                  ORDER BY total_count DESC
                  LIMIT 10
                  """,conn)

,name,total_count,average_rating
0,McDonald's,16490,1.868702
1,Chipotle Mexican Grill,9071,2.381757
2,Taco Bell,8017,2.141813
3,Chick-fil-A,7687,3.377419
4,First Watch,6761,3.875000
5,Panera Bread,6613,2.661905
6,Buffalo Wild Wings,6483,2.344828
7,Domino's Pizza,6091,2.290210
8,Wendy's,5930,2.030159
9,Chili's,5744,2.514706


In [16]:
# which restautrants have the highest rating

pd.read_sql_query(f"""SELECT name,
                  SUM(review_count) as total_count,
                  avg(stars) as average_rating
                  FROM yelp_academic_dataset_business
                  WHERE business_id IN {tuple(business_id['business_id'])}
                  GROUP BY name
                  ORDER BY average_rating DESC
                  LIMIT 10
                  """,conn)

,name,total_count,average_rating
0,ā café,48,5.0
1,two birds cafe,77,5.0
2,the brewers cabinet production,13,5.0
3,taqueria la cañada,17,5.0
4,la bamba,44,5.0
5,la 5th av tacos,24,5.0
6,el sabor mexican and chinese food,21,5.0
7,eat.drink.Om...YOGA CAFE,7,5.0
8,d4 Tabletop Gaming Cafe,8,5.0
9,cabbage vegetarian cafe,12,5.0


In [17]:
# Do restaurants with higher engagement tend to have higher ratings?

pd.read_sql_query("""
                  SELECT business_id,
                  SUM(length(date) - length(replace(date,',',''))+1) as total_checkins
                  FROM yelp_academic_dataset_checkin
                  GROUP BY business_id
                  """,conn)

,business_id,total_checkins
0,---kPU91CF4Lq2-WlRu9Lw,11
1,--0iUa4sNDFiZFrAdIWhZQ,10
2,--30_8IhuyMHbSOcNWd6DQ,2
3,--7PUidqRWpRSpXebiyxTg,10
4,--7jw19RH9JKXgFohspgQw,26
...,...,...
131925,zznJox6-nmXlGYNWgTDwQQ,67
131926,zznZqH9CiAznbkV6fXyHWA,1
131927,zzu6_r3DxBJuXcjnOYVdTw,23
131928,zzw66H6hVjXQEt0Js3Mo4A,2


In [18]:
# tip count
pd.read_sql_query("""
                  SELECT business_id,
                  count(business_id) as total_tips
                  FROM yelp_academic_dataset_tip
                  GROUP BY business_id
                  """,conn)

,business_id,total_tips
0,---kPU91CF4Lq2-WlRu9Lw,4
1,--0iUa4sNDFiZFrAdIWhZQ,6
2,--30_8IhuyMHbSOcNWd6DQ,1
3,--7PUidqRWpRSpXebiyxTg,3
4,--8IbOsAAxjKRoYsBFL-PA,4
...,...,...
106188,zzjCxn89a7RQo8keIOO_Ag,1
106189,zzjFdJwXuxBOGe9JeY_EMw,2
106190,zznJox6-nmXlGYNWgTDwQQ,6
106191,zzu6_r3DxBJuXcjnOYVdTw,2


In [19]:
conn.execute("""
CREATE INDEX IF NOT EXISTS idx_checkin_business
ON yelp_academic_dataset_checkin(business_id)
""")
conn.execute("""
CREATE INDEX IF NOT EXISTS idx_tip_business
ON yelp_academic_dataset_tip(business_id)
""")
conn.execute("""
CREATE INDEX IF NOT EXISTS idx_business_business
ON yelp_academic_dataset_business(business_id)
""")

In [ ]:
# query = f""" SELECT
#                     total.average_rating as rating,
#                     AVG(total.total_review_count) as average_review_count,
#                     AVG(total.total_checkins) as average_checkins,
#                     AVG(total.total_tips) as average_tips
#                   FROM
#                   (SELECT 
#                     b.business_id,
#                     SUM(b.review_count) as total_review_count,
#                     avg(b.stars) as average_rating,
#                     SUM(length(cc.date) - length(replace(cc.date,',',''))+1) as total_checkins,
#                      SUM(tip.total_tips) as total_tips
#                   FROM yelp_academic_dataset_business b
#                   LEFT JOIN yelp_academic_dataset_checkin cc
#                   ON b.business_id = cc.business_id
#                   LEFT JOIN (SELECT business_id,
#                                     count(business_id) as total_tips
#                                     FROM yelp_academic_dataset_tip
#                                     GROUP BY business_id) tip
#                   WHERE b.business_id IN {tuple(business_id['business_id'])}
#                   GROUP BY b.business_id) as total
#                   GROUP BY total.average_rating """
# for chunk in pd.read_sql_query(query,conn,chunksize=1000):
#           print(chunk.shape)

In [21]:
query = f"""
WITH selected_businesses AS(
    SELECT business_id
    FROM yelp_academic_dataset_business b
    WHERE business_id IN {tuple(business_id['business_id'])}
),
checkin_totals AS(
    SELECT c.business_id,
    SUM(LENGTH(date) - LENGTH(REPLACE(date,',','')+1)) as total_checkins
    FROM yelp_academic_dataset_checkin c
    INNER JOIN selected_businesses sb
        ON c.business_id = sb.business_id
    GROUP BY c.business_id
),
tip_totals AS(
    SELECT t.business_id,
    COUNT(t.business_id) as total_tips
    FROM yelp_academic_dataset_tip t
    INNER JOIN selected_businesses sb
        ON t.business_id = sb.business_id
    GROUP BY t.business_id
)

SELECT
    b.stars as rating,
    AVG(b.review_count) as average_review_count,
    AVG(COALESCE(ct.total_checkins,0)) as average_checkins,
    AVG(COALESCE(tt.total_tips,0)) as average_tips
FROM selected_businesses sb

JOIN yelp_academic_dataset_business b
    ON sb.business_id = b.business_id
LEFT JOIN checkin_totals ct
    ON sb.business_id = ct.business_id
LEFT JOIN tip_totals tt
    ON sb.business_id = tt.business_id
GROUP BY b.stars  
"""

for chunk in pd.read_sql_query(query,conn,chunksize=1000):
    print(chunk)

   rating  average_review_count  average_checkins  average_tips
0     1.0             14.365079        317.841270      1.751323
1     1.5             24.358459        695.246231      3.243719
2     2.0             27.759629       1069.238979      3.815777
3     2.5             36.631037       1638.421120      5.441404
4     3.0             48.054998       2190.509356      7.384984
5     3.5             63.730125       2611.562227      9.512747
6     4.0             73.136954       2638.516291     10.328477
7     4.5             65.282554       1781.525678      7.931141
8     5.0             31.127979        537.585172      2.912621
